# Part 2 - Submitting Python Jobs on the SCC: CPU vs. GPU

Part 1 introduced submitting a basic batch job with `qsub`. This notebook covers the details you need for real workloads: **requesting the right resources**, the differences between **CPU** and **GPU** job scripts, and monitoring/managing jobs. This material is meant to be followed on the SCC itself.

## Anatomy of a Batch Script
Every SGE batch script is a shell script with `#$` directive lines. Common directives:

| Directive | Meaning |
|---|---|
| `-N name` | Job name |
| `-l h_rt=HH:MM:SS` | Maximum wall-clock run time |
| `-pe omp N` | Request N CPU cores (shared-memory parallel environment) |
| `-j y` | Merge stdout/stderr into one log file |
| `-m e` | Email when the job ends |
| `-l gpus=1` | Request 1 GPU |
| `-l gpu_c=X.X` | Minimum GPU compute capability |

Run `man qsub` or see the SCC documentation for the full list.

## Example 1: A CPU Job

`cpu_job.qsub`:
```bash
#!/bin/bash -l

#$ -N numpy_benchmark
#$ -l h_rt=00:30:00
#$ -pe omp 4          # Request 4 CPU cores
#$ -j y


module load miniconda
source activate energize

# Encourage NumPy's BLAS libraries to use the 4 allocated CPU cores.
export OMP_NUM_THREADS=4
export MKL_NUM_THREADS=4
export OPENBLAS_NUM_THREADS=4

python benchmark.py
```

Submit it from the directory containing `cpu_job.qsub` and `benchmark.py`:
```bash
qsub cpu_job.qsub
```

The thread variables encourage NumPy's underlying BLAS library to parallelize matrix multiplication across the four allocated cores. The actual behavior depends on the BLAS library installed in the environment.

## Example 2: A GPU Job

GPU jobs need two extra things: a GPU resource request, and (usually) a CUDA/GPU-enabled module.

`gpu_job.qsub`:
```bash
#!/bin/bash -l

#$ -N gpu_benchmark
#$ -l h_rt=00:30:00
#$ -l gpus=1                 # Request 1 GPU
#$ -l gpu_c=7.0               # Minimum compute capability
#$ -j y

module load miniconda
module load cuda/12.8
conda activate energize

python gpu_benchmark.py
```

Submit the same way:
```bash
qsub gpu_job.qsub
```

Use `qgpus` to see what GPU models are currently available on the cluster, and their current usage.

## Interactive Sessions
For quick testing (rather than a full batch submission), you can request an **interactive session** on a compute node. This drops you into a shell on a compute node so you can run Python directly, with a live terminal:

```bash
qrsh -l h_rt=01:00:00 -pe omp 4
```

For an interactive session with a GPU:
```bash
qrsh -l h_rt=01:00:00 -l gpus=1
```

Interactive sessions are great for debugging, but for anything long-running or for many similar runs, use a batch job (`qsub`) so it can run unattended and the scheduler can manage resources fairly across all users.

## Monitoring and Managing Jobs

| Command | Purpose |
|---|---|
| `qstat -u your_username` | List your queued/running jobs |
| `qstat -j <job_id>` | Detailed status of a specific job |
| `qdel <job_id>` | Cancel a job |
| `qacct -j <job_id>` | Resource usage report for a finished job (CPU time, memory, wall time) |

Checking `qacct` after a job finishes is a good habit - it tells you whether you requested too much (wasting your allocation) or too little (risking your job being killed for exceeding its time/memory limit) for next time.

## Array Jobs: Running Many Similar Jobs at Once
If you need to run the same script many times with different inputs (e.g., one run per material sample), an **array job** submits them all as a single job with a `-t` range, and the special environment variable `$SGE_TASK_ID` tells each task which input to use.

`array_job.qsub`:
```bash
#!/bin/bash -l
#$ -N array_demo
#$ -l h_rt=00:10:00
#$ -t 1-6
#$ -j y

module load python3/3.12.4

# SGE_TASK_ID will be 1, 2, 3, ... 6 for each task in the array
python process_sample.py $SGE_TASK_ID
```

### *Exercise*
1. Write a CPU batch script that requests 2 hours and 8 cores, and runs `python train_model.py --epochs 50`.
2. Modify it to request 1 GPU instead of extra CPU cores.
3. Submit the GPU benchmark script from the previous notebook (Part 2 Notebook 5) as a batch job, then check its status with `qstat` and its resource usage with `qacct` once it finishes.